# Experiment 5 — Random demand and remaining collapse time
Run all in Colab. CPU only. We retain the whole-step strict-overload convention.

## Mathematical specification
Let $S_s=(A_s,L_s,U_s)$ be the settled state: active nodes, loads, and cumulative unserved demand. With independent $X_s\sim\mathrm{Uniform}(1-a,1+a)$, $0\le a<1$,

$$S_{s+1}=\Phi(S_s,X_s),\qquad \Phi(S,x)=\mathcal R_G(A,L+x/|A|,U).$$

Here $\mathcal R_G$ resolves all synchronous overload rounds using the specified capacities. Total-load accounting holds for each sample path. Define $B=\{S:N-|A|\ge\lceil .3N\rceil\}$ and the remaining collapse time $R=\inf\{r\ge0:S_{s+r}\in B\}$.

The **exact distribution equation** is a first-passage recurrence:

$$F_S(0)=\mathbf1_{S\in B},\qquad
F_S(h)=\begin{cases}1&S\in B,\\
\displaystyle\frac1{2a}\int_{1-a}^{1+a}F_{\Phi(S,x)}(h-1)\,dx&S\notin B,
\end{cases}$$

for integer $h\ge1$ and $a>0$. At $a=0$ replace the integral by evaluation at $x=1$. Thus $P(R=h)=F_S(h)-F_S(h-1)$. This is a concrete recursive equation; its high-dimensional state makes direct integration expensive. We approximate it by independent Monte Carlo continuations:

$$\widehat F_S(h)=M^{-1}\sum_{m=1}^M\mathbf1_{R_m\le h}.$$

The approximate Monte Carlo standard error at a fixed horizon is $\sqrt{\widehat F(1-\widehat F)/M}$. This measures simulation sampling error, not model uncertainty. Uncollapsed paths at the forecast limit remain censored; do not label their limit as a collapse time.

This first-passage formulation is our specialization of standard Markov hitting-time reasoning, not a claim of a novel general theorem. [Reference: Grinstead & Snell, Introduction to Probability, chapter 11](https://math.dartmouth.edu/~prob/prob/prob.pdf). Overload-cascade background: [Motter & Lai](https://arxiv.org/abs/cond-mat/0301086); our local load rule differs from theirs.

Because demand is shared equally, this extension perturbs demand magnitude rather than routing. Whole-step overshoot can also affect cascades, so it is not claimed to be only a continuous-time clock change.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

SEED = 42
N = 100
MEAN_DEGREE = 6
CAPACITY = 10.0
INITIAL_UTILIZATION = (0.4, 0.7)
DEMAND_INCREMENT = 1.0
COLLAPSE_FRACTION = 0.30
MAX_STEPS = 1000

In [ ]:
def resolve_cascade(graph, loads, capacities, active, unserved=0.0):
    """Return a settled copy and a trace of synchronous failure rounds."""
    loads = dict(loads)
    active = set(active)
    trace = []
    while True:
        overloaded = {i for i in active if loads[i] > capacities[i]}
        if not overloaded:
            break
        survivors = active - overloaded
        additions = {i: 0.0 for i in survivors}
        for i in sorted(overloaded):
            recipients = sorted(set(graph.neighbors(i)) & survivors)
            if recipients:
                share = loads[i] / len(recipients)
                for j in recipients:
                    additions[j] += share
            else:
                unserved += loads[i]
            loads[i] = 0.0
        for j, amount in additions.items():
            loads[j] += amount
        active = survivors
        trace.append({'active': set(active), 'loads': dict(loads),
                      'failed': sorted(overloaded), 'unserved': unserved})
    return loads, active, unserved, trace

In [ ]:
def simulate(seed=SEED, n=N, mean_degree=MEAN_DEGREE,
             capacity=CAPACITY, utilization=INITIAL_UTILIZATION,
             delta_d=DEMAND_INCREMENT, collapse_fraction=COLLAPSE_FRACTION,
             max_steps=MAX_STEPS, capacity_spread=0.0):
    if not (n >= 2 and 0 < mean_degree <= n-1 and capacity > 0
            and 0 <= utilization[0] <= utilization[1] < 1
            and delta_d > 0 and 0 < collapse_fraction <= 1 and max_steps >= 1):
        raise ValueError('Invalid simulation settings')
    if n % 2 or not 0 <= capacity_spread <= 0.3:
        raise ValueError('This paired-capacity experiment requires even N and spread between 0 and 0.3')
    rng = np.random.default_rng(seed)
    for attempt in range(10000):
        graph = nx.gnp_random_graph(n, mean_degree/(n-1),
                                   seed=int(rng.integers(0, 2**32-1)))
        if nx.is_connected(graph):
            break
    else:
        raise RuntimeError('Could not sample connected graph; revise connectivity')
    capacities = dict.fromkeys(graph, float(capacity))
    loads = {i: capacity * rng.uniform(*utilization) for i in graph}
    # Separate RNG preserves identical graph and initial loads for each paired seed.
    capacity_rng = np.random.default_rng(np.random.SeedSequence([seed, 2026]))
    offsets = capacity_rng.uniform(0, 1, n//2)
    offsets = np.concatenate([offsets, -offsets])
    capacity_rng.shuffle(offsets)
    capacities = {i: capacity*(1+capacity_spread*offsets[i]) for i in graph}
    assert np.isclose(sum(capacities.values()), n*capacity)
    assert all(loads[i] < capacities[i] for i in graph)
    active = set(graph)
    initial = sum(loads.values())
    unserved = injected = 0.0
    history, frames = [], []
    collapse_step = None

    def frame(step, phase, failed=()):
        frames.append(dict(step=step, phase=phase, active=set(active),
                           loads=dict(loads), failed=list(failed)))

    def record(step, cascade_size):
        active_total = sum(loads[i] for i in active)
        assert np.isclose(active_total + unserved, initial + injected)
        spare = sum(capacities[i] - loads[i] for i in active)
        near = sum(loads[i]/capacities[i] >= .9 for i in active)
        giant = max((len(c) for c in nx.connected_components(graph.subgraph(active))), default=0)
        history.append(dict(step=step, injected=injected, failed=n-len(active),
                            failed_fraction=1-len(active)/n, spare=spare/(n*capacity),
                            near_fraction=near/len(active) if active else np.nan,
                            unserved=unserved, active_load=active_total,
                            cascade_size=cascade_size, giant_fraction=giant/n))

    record(0, 0)
    frame(0, 'initial')
    for step in range(1, max_steps+1):
        before = len(active)
        for i in active:
            loads[i] += delta_d / before
        injected += delta_d
        frame(step, 'demand added')
        loads, active, unserved, trace = resolve_cascade(
            graph, loads, capacities, active, unserved)
        for round_number, state in enumerate(trace, 1):
            assert np.isclose(sum(state['loads'][i] for i in state['active'])
                              + state['unserved'], initial + injected)
            frames.append(dict(step=step, phase=f'cascade round {round_number}',
                               active=state['active'], loads=state['loads'],
                               failed=state['failed']))
        record(step, before-len(active))
        if (n-len(active)) >= collapse_fraction*n - 1e-9:
            collapse_step = step
            break
    return dict(graph=graph, capacities=capacities, history=history,
                frames=frames, collapse_step=collapse_step,
                initial_load=initial, attempts=attempt+1)


In [ ]:
def transition(graph, capacities, state, demand):
    loads, active, unserved = state
    loads = dict(loads)
    if not active: return loads, set(), unserved
    before = sum(loads[i] for i in active)+unserved
    for i in active: loads[i] += demand/len(active)
    loads, active, unserved, _ = resolve_cascade(graph, loads, capacities, active, unserved)
    assert np.isclose(sum(loads[i] for i in active)+unserved, before+demand)
    return loads, active, unserved

def collapsed(state, n):
    return n-len(state[1]) >= int(np.ceil(COLLAPSE_FRACTION*n))

def remaining_time(graph, capacities, state, amplitude, rng, horizon=1000):
    if collapsed(state, len(graph)): return 0
    for step in range(1,horizon+1):
        x = 1.0 if amplitude==0 else rng.uniform(1-amplitude,1+amplitude)
        state = transition(graph, capacities, state, x)
        if collapsed(state,len(graph)): return step
    return None

setup = simulate(seed=1000, capacity_spread=.3)
graph, capacities = setup['graph'], setup['capacities']
initial = (dict(setup['frames'][0]['loads']), set(graph), 0.)
expected = setup['collapse_step']
assert remaining_time(graph, capacities, initial, 0., np.random.default_rng(1)) == expected
# A forecast uses only a present state; independent RNG streams represent future paths.
AMPLITUDES = (0., .2, .5)
M = 300
HORIZON = 1000
OBSERVATION_STEP = 20
observed_states, forecasts = {}, {}
for a in AMPLITUDES:
    observed_rng = np.random.default_rng(20261030)
    state = initial
    for step in range(OBSERVATION_STEP):
        state = transition(graph,capacities,state,1.0)
        if collapsed(state,len(graph)): break
    if collapsed(state,len(graph)):
        print(a, 'already collapsed before observation; no pre-collapse forecast')
        continue
    observed_states[a] = state
    draws = []
    for child in np.random.SeedSequence([2026, int(a*1000)]).spawn(M):
        draws.append(remaining_time(graph,capacities,state,a,np.random.default_rng(child),HORIZON))
    forecasts[a] = draws
    finite = np.array([r for r in draws if r is not None])
    print(f'a={a}: {len(finite)}/{M} observed collapses within horizon')
    if len(finite)==M:
        print('Remaining steps: median, 10th percentile, 90th percentile:',np.quantile(finite,[.5,.1,.9],method='inverted_cdf'))
    else:
        print('Censoring present: finite-only quantiles would be conditional and are not reported.')
    for h in (10,25,50,100):
        p = sum(r is not None and r<=h for r in draws)/M
        print(f'P(collapse within {h} steps) ≈ {p:.3f}; MC standard error ≈ {np.sqrt(p*(1-p)/M):.3f}')
print('Zero-variability check matched original deterministic solver.')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
for a, draws in forecasts.items():
    finite = [r for r in draws if r is not None]
    if finite:
        axes[0].hist(finite,bins='auto',histtype='step',linewidth=2,label=f'a={a}')
        x = np.arange(0,max(finite)+1)
        axes[1].step(x,[sum(r is not None and r<=h for r in draws)/M for h in x],where='post',label=f'a={a}')
axes[0].set(xlabel='Steps remaining',ylabel='Number of forecast paths',title='Possible remaining collapse times')
axes[1].set(xlabel='Future horizon (steps)',ylabel='Estimated probability of collapse',ylim=(0,1.02),title='Cumulative collapse probability')
for ax in axes: ax.legend(); ax.grid(alpha=.2)
fig.tight_layout();plt.show()

## Interpretation and limitations

These are conditional distributions for one network and one observation time, not calibrated real-world predictions. A repeated independent-future coverage test and comparisons across graphs remain necessary. All amplitudes share the identical present state after 20 constant-demand steps. Only future demand variability changes. A sample probability of zero or one does not prove that the true probability is exactly zero or one; the simple standard-error formula is unreliable at these boundaries.

Model parameters: graph, capacities, initial loads, mean demand 1, variability amplitude a, collapse threshold. Numerical settings: M paths, forecast horizon, observation step. There is no fitted predictor. Increase M to reduce Monte Carlo error, not to change the assumed physical uncertainty. Save notebook/CSV results before terminating Colab runtimes.